# NodPT Fine-Tuning Pipeline

This notebook walks through the complete fine-tuning workflow for NodPT node-type models:

1. **Generate Training Samples** (`scripts/generate_samples.py`) — use a large TensorRT-LLM-served model (70B+) to produce high-quality JSONL training data.
2. **Fine-Tune** (`scripts/finetune.py`) — fine-tune a smaller base model with Unsloth FP4 (4-bit) precision using LoRA adapters.
3. **Export to GGUF** (`scripts/export_gguf.py`) — merge LoRA weights and export to GGUF format for local deployment via Ollama.

---

## Node Hierarchy

NodPT decomposes project requests through a four-level node hierarchy:

| Node Type  | Receives              | Produces                                         |
|------------|-----------------------|--------------------------------------------------|
| Director   | Project request       | `content` + `managers[]` (name, job)             |
| Manager    | Job from Director     | `content` + `supervisors[]` (name, job)          |
| Supervisor | Job from Manager      | `content` + `agents[]` (name, job)               |
| Agent      | Job from Supervisor   | `content` + `files[]` (filename, content)        |

Each node type has a JSON Schema in `AI/src/<NodeType>/format.json`. Fine-tuning teaches the model to produce valid JSON that matches these schemas.

---

## Prerequisites

- Python 3.10+
- NVIDIA GPU with ≥16 GB VRAM and CUDA 11.8+
- Ollama installed locally (for deployment)

```bash
# Install fine-tuning dependencies
cd AI/src/fine-tuning
pip install -r requirements.txt
```

---
## Part 1 — Generate Training Samples

**Script:** `AI/src/fine-tuning/scripts/generate_samples.py`

This script uses a large model (70B+) served by **TensorRT-LLM** to generate high-quality JSONL training data for each NodPT node type. TensorRT-LLM exposes an OpenAI-compatible `/v1/chat/completions` endpoint and uses **continuous batching**, so multiple concurrent requests are processed efficiently server-side.

### How it works
1. Builds a structured prompt instructing the large model to output JSONL training lines.
2. Fires concurrent async requests (via `aiohttp`) up to a configurable concurrency limit.
3. Parses and validates each generated sample against the node type's schema.
4. Discards invalid samples and retries until the requested count is reached.
5. Appends valid samples to `data-samples/<node-type>.jsonl`, deduplicating by input.

### 1.1 Imports and Configuration

Standard library imports plus `aiohttp` for async HTTP. All key constants are defined at module level so they can be overridden without touching business logic.

In [ ]:
import argparse
import asyncio
import json
import os
import sys
import time

import aiohttp

# ── Path resolution ───────────────────────────────────────────────────────────
# Assumes this notebook is run from AI/ or AI/src/fine-tuning/scripts/.
# Adjust BASE_DIR if running from a different working directory.
SCRIPT_DIR = os.path.abspath("AI/src/fine-tuning/scripts")
BASE_DIR   = os.path.abspath("AI/src/fine-tuning")
AI_SRC_DIR = os.path.abspath("AI/src")
DATA_DIR   = os.path.join(BASE_DIR, "data-samples")

# ── Supported node types ──────────────────────────────────────────────────────
NODE_TYPES = ["director", "manager", "supervisor", "agent"]

# ── TensorRT-LLM endpoint defaults ───────────────────────────────────────────
DEFAULT_ENDPOINT    = "http://localhost:8000"
DEFAULT_MODEL       = "meta-llama/Llama-3.1-70B-Instruct"
DEFAULT_CONCURRENCY = 8    # Max simultaneous requests to TensorRT-LLM
DEFAULT_BATCH_SIZE  = 5    # Samples requested per API call

# ── Safety limits ────────────────────────────────────────────────────────────
MAX_STALL_ITERATIONS    = 5     # Stop if no new samples after N consecutive rounds
MAX_GENERATION_SECONDS  = 1800  # 30-minute hard cap per node type

print("Configuration loaded successfully.")
print(f"Data directory: {DATA_DIR}")

### 1.2 Node Type Configuration

`NODE_CONFIG` centralises per-node-type metadata:
- `instruction` — the system instruction string used in every training sample for this node type.
- `array_field` — the key in the output JSON that holds the list of sub-items.
- `item_fields` — required fields for each item in that list.
- `min_items` / `max_items` — validation constraints on the list length (2–5 items).

In [ ]:
NODE_CONFIG = {
    "director": {
        "instruction": (
            "You are a Director AI. Analyze the following project request "
            "and break it down into manager assignments. For each manager, "
            "provide a clear name and job description."
        ),
        "array_field": "managers",
        "item_fields": ["name", "job"],
        "min_items": 2,
        "max_items": 5,
    },
    "manager": {
        "instruction": (
            "You are a Manager AI. Analyze the following job assigned by "
            "the Director and break it down into supervisor assignments. "
            "For each supervisor, provide a clear name and job description."
        ),
        "array_field": "supervisors",
        "item_fields": ["name", "job"],
        "min_items": 2,
        "max_items": 5,
    },
    "supervisor": {
        "instruction": (
            "You are a Supervisor AI. Analyze the following job assigned by "
            "the Manager and break it down into agent assignments. For each "
            "agent, provide a clear name and job description."
        ),
        "array_field": "agents",
        "item_fields": ["name", "job"],
        "min_items": 2,
        "max_items": 5,
    },
    "agent": {
        "instruction": (
            "You are an Agent AI. Complete the following job and produce "
            "the file output. For each file, provide the filename and "
            "the full content."
        ),
        "array_field": "files",
        "item_fields": ["filename", "content"],
        "min_items": 2,
        "max_items": 5,
    },
}

# Quick sanity check — verify all four node types are configured
assert set(NODE_CONFIG.keys()) == set(NODE_TYPES), "NODE_CONFIG must cover all NODE_TYPES"
print("NODE_CONFIG validated for all node types:", NODE_TYPES)

### 1.3 Schema Loader

Loads the Ollama-compatible JSON Schema from `AI/src/<NodeType>/format.json`. These schemas are also used at inference time to constrain Ollama's output — fine-tuning teaches the model to respect them without the constraint.

In [ ]:
def load_format_schema(node_type: str) -> dict:
    """
    Load the format.json schema for a given node type.

    Args:
        node_type: One of 'director', 'manager', 'supervisor', 'agent'.

    Returns:
        Parsed JSON Schema dict from AI/src/<NodeType>/format.json.

    Raises:
        FileNotFoundError: if the schema file does not exist.
    """
    type_name = node_type.capitalize()
    path = os.path.join(AI_SRC_DIR, type_name, "format.json")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# Preview the Director schema to confirm paths are correct
director_schema = load_format_schema("director")
print("Director schema top-level keys:", list(director_schema.keys()))
print("Required fields:", director_schema.get("required"))

### 1.4 Output Validation

`validate_sample_output` checks that a model-generated output string satisfies the node type's schema:
- Must be valid JSON representing an object.
- Must have a non-empty `content` string field.
- Must have the node-type-specific array field (e.g., `managers`) with between `min_items` and `max_items` entries.
- Each array item must be an object with all required fields filled in (no empty strings).

Returns `(True, "valid")` or `(False, reason_string)` so callers can report why a sample was rejected.

In [ ]:
def validate_sample_output(output_str: str, node_type: str):
    """
    Validate a generated output string against the node type schema.

    Args:
        output_str: JSON string produced by the large model.
        node_type:  One of 'director', 'manager', 'supervisor', 'agent'.

    Returns:
        Tuple (is_valid: bool, reason: str).
    """
    config = NODE_CONFIG[node_type]

    # ── Step 1: Ensure it is parseable JSON ──────────────────────────────────
    try:
        obj = json.loads(output_str)
    except json.JSONDecodeError:
        return False, "output is not valid JSON"

    if not isinstance(obj, dict):
        return False, "output is not a JSON object"

    # ── Step 2: Validate 'content' field ──────────────────────────────────────
    if "content" not in obj or not isinstance(obj["content"], str):
        return False, "missing or invalid 'content' field"
    if not obj["content"].strip():
        return False, "'content' field is empty"

    # ── Step 3: Validate the node-type array field ────────────────────────────
    array_field = config["array_field"]
    if array_field not in obj or not isinstance(obj[array_field], list):
        return False, f"missing or invalid '{array_field}' field"

    min_items = config.get("min_items", 2)
    max_items = config.get("max_items", 5)
    length = len(obj[array_field])
    if length < min_items:
        return False, f"'{array_field}' array has {length} items; expected at least {min_items}"
    if length > max_items:
        return False, f"'{array_field}' array has {length} items; expected at most {max_items}"

    # ── Step 4: Validate each array item ──────────────────────────────────────
    for i, item in enumerate(obj[array_field]):
        if not isinstance(item, dict):
            return False, f"{array_field}[{i}] is not an object"
        for field in config["item_fields"]:
            if field not in item or not isinstance(item[field], str):
                return False, f"{array_field}[{i}] missing or invalid '{field}'"
            if not item[field].strip():
                return False, f"{array_field}[{i}].{field} is empty"

    return True, "valid"


# ── Quick inline tests ────────────────────────────────────────────────────────
# Valid director output
valid_output = json.dumps({
    "content": "I'll split the project into backend and frontend work.",
    "managers": [
        {"name": "Backend Manager", "job": "Build the REST API."},
        {"name": "Frontend Manager", "job": "Build the web UI."},
    ],
})
ok, msg = validate_sample_output(valid_output, "director")
print(f"Valid director sample: ok={ok}, msg='{msg}'")

# Invalid — missing content
invalid_output = json.dumps({"managers": [{"name": "M", "job": "J"}, {"name": "M2", "job": "J2"}]})
ok, msg = validate_sample_output(invalid_output, "director")
print(f"Missing content field: ok={ok}, msg='{msg}'")

### 1.5 Prompt Builder

`build_generation_prompt` constructs the user-facing prompt sent to the large model. It asks the model to produce `batch_size` JSONL lines in a single response. An optional `existing_inputs` list is appended so the model avoids regenerating duplicate samples — critical when calling the endpoint many times in a long generation run.

In [ ]:
def build_generation_prompt(
    node_type: str,
    batch_size: int,
    existing_inputs: list | None = None,
) -> str:
    """
    Build the prompt sent to the large model to generate JSONL training samples.

    Args:
        node_type:       Target node type ('director', 'manager', 'supervisor', 'agent').
        batch_size:      Number of JSONL lines to generate in one API call.
        existing_inputs: Previously seen input strings to exclude from this batch.

    Returns:
        Fully formatted prompt string.
    """
    config = NODE_CONFIG[node_type]
    array_field = config["array_field"]

    # Human-readable description of the expected array items
    item_desc = (
        "each with 'filename' (string) and 'content' (string, the full file content)"
        if node_type == "agent"
        else "each with 'name' (string) and 'job' (string)"
    )

    # Deduplication hint — show up to 20 existing inputs to avoid prompt bloat
    avoid_section = ""
    if existing_inputs:
        sample_list = existing_inputs[:20]
        avoid_section = (
            "\n\nDo NOT reuse these existing inputs (generate completely different ones):\n"
            + "\n".join(f"- {inp}" for inp in sample_list)
        )

    # Role and input description tables used to vary the language
    node_role = {
        "director":   "a Director AI that breaks down project requests into manager assignments",
        "manager":    "a Manager AI that breaks down a job into supervisor assignments",
        "supervisor": "a Supervisor AI that breaks down a job into agent assignments",
        "agent":      "an Agent AI that completes a job by producing source code files",
    }
    input_desc = {
        "director":   "a realistic software project request (1-2 sentences describing what to build)",
        "manager":    "a realistic job description assigned by a Director (specific technical domain to manage)",
        "supervisor": "a realistic job description assigned by a Manager (specific technical task area to supervise)",
        "agent":      "a realistic job description assigned by a Supervisor (specific coding task to implement)",
    }

    prompt = (
        f"Generate exactly {batch_size} high-quality training samples for fine-tuning {node_role[node_type]}.\n\n"
        f"Each sample must be a JSON object on its own line (JSONL format) with these fields:\n"
        f'- "instruction": exactly "{config["instruction"]}"\n'
        f"- \"input\": {input_desc[node_type]}\n"
        f'- "output": a JSON string containing a valid response with:\n'
        f'  - "content": a 1-3 sentence explanation of the plan/work\n'
        f'  - "{array_field}": an array of 2-5 items, {item_desc}\n\n'
        f"Rules:\n"
        f"1. Each input must be UNIQUE and cover different software domains (web, mobile, backend, data, DevOps, ML, etc.).\n"
        f"2. The \"output\" field must be a valid JSON STRING (escaped properly to be embedded in the JSONL line).\n"
        f'3. Each item in "{array_field}" must have meaningful, specific descriptions — not generic placeholders.\n'
        f"4. For Agent type: files must contain realistic code with proper syntax for the language used.\n"
        f'5. Vary the number of items in "{array_field}" between 2 and 5 across samples.\n'
        f"6. Inputs should be diverse: different industries, tech stacks, and complexity levels.\n"
        f"{avoid_section}\n\n"
        f"Output ONLY the {batch_size} JSONL lines, one per line. No markdown, no explanation, no code fences."
    )
    return prompt


# Preview the first 300 characters of a director prompt
sample_prompt = build_generation_prompt("director", 3)
print(sample_prompt[:400])
print(f"\n... (total length: {len(sample_prompt)} chars)")

### 1.6 TensorRT-LLM API Call

`call_tensorrt` sends a single chat completion request to the TensorRT-LLM OpenAI-compatible endpoint. Key design points:
- Uses `aiohttp` for non-blocking I/O so multiple requests can be in-flight simultaneously.
- Hard timeout of 300 s per request.
- Returns `(content, None)` on success or `(None, error_message)` on failure — callers handle errors without raising exceptions.

In [ ]:
async def call_tensorrt(
    session: aiohttp.ClientSession,
    endpoint: str,
    model: str,
    prompt: str,
    temperature: float = 0.8,
):
    """
    Send one chat-completion request to the TensorRT-LLM OpenAI-compatible API.

    Args:
        session:     Shared aiohttp session.
        endpoint:    Base URL of the TensorRT-LLM server, e.g. 'http://localhost:8000'.
        model:       HuggingFace model ID served by TensorRT-LLM.
        prompt:      User message to send.
        temperature: Sampling temperature (higher → more diverse, lower → more focused).

    Returns:
        Tuple (response_text: str | None, error: str | None).
    """
    url = f"{endpoint}/v1/chat/completions"
    payload = {
        "model": model,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a training data generator. You produce JSONL lines "
                    "for fine-tuning language models. Output ONLY valid JSONL — "
                    "one JSON object per line, no markdown, no extra text."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        "temperature": temperature,
        "max_tokens": 4096,
        "stream": False,
    }

    try:
        async with session.post(
            url,
            json=payload,
            headers={"Content-Type": "application/json"},
            timeout=aiohttp.ClientTimeout(total=300),
        ) as resp:
            if resp.status != 200:
                text = await resp.text()
                return None, f"HTTP {resp.status}: {text[:200]}"
            data = await resp.json()
            content = data["choices"][0]["message"]["content"]
            return content, None
    except asyncio.TimeoutError:
        return None, "request timed out (300s)"
    except aiohttp.ClientError as e:
        return None, f"connection error: {e}"


print("call_tensorrt defined. Requires a running TensorRT-LLM server to execute.")

### 1.7 Response Parser

`parse_generated_lines` processes the raw multi-line text returned by the large model:
- Skips blank lines and Markdown code-fence markers (`` ``` ``).
- Parses each line as JSON; skips lines that are not valid JSON.
- Validates the presence of required JSONL keys (`instruction`, `input`, `output`).
- Auto-converts `output` from object to JSON string if the model forgot to stringify it.
- Runs `validate_sample_output` to enforce schema compliance.
- Overwrites `instruction` with the canonical value from `NODE_CONFIG` for consistency.

In [ ]:
def parse_generated_lines(raw_text: str, node_type: str):
    """
    Parse and validate raw model output into a list of JSONL training samples.

    Args:
        raw_text:  Multi-line string returned by the large model.
        node_type: Node type to validate output structure against.

    Returns:
        Tuple (valid_samples: list[dict], invalid_count: int).
    """
    valid_samples = []
    invalid_count = 0

    for line in raw_text.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        # Discard Markdown code-fence lines (``` or ```json)
        if line.startswith("```"):
            continue

        # ── Attempt JSON parse ──────────────────────────────────────────────
        try:
            sample = json.loads(line)
        except json.JSONDecodeError:
            invalid_count += 1
            continue

        # ── Require all three JSONL keys ────────────────────────────────────
        if not all(k in sample for k in ("instruction", "input", "output")):
            invalid_count += 1
            continue

        if not isinstance(sample["instruction"], str) or not sample["instruction"].strip():
            invalid_count += 1
            continue

        if not isinstance(sample["input"], str) or not sample["input"].strip():
            invalid_count += 1
            continue

        # ── Auto-convert output object → JSON string ────────────────────────
        output_str = sample["output"]
        if not isinstance(output_str, str):
            try:
                output_str = json.dumps(sample["output"], ensure_ascii=False)
                sample["output"] = output_str
            except (TypeError, ValueError):
                invalid_count += 1
                continue

        # ── Validate output against node type schema ────────────────────────
        ok, reason = validate_sample_output(output_str, node_type)
        if not ok:
            invalid_count += 1
            continue

        # Normalise instruction to the canonical value
        sample["instruction"] = NODE_CONFIG[node_type]["instruction"]
        valid_samples.append(sample)

    return valid_samples, invalid_count


# ── Demo: parse a mix of valid and invalid lines ──────────────────────────────
demo_valid_line = json.dumps({
    "instruction": "old instruction",
    "input": "Build a real-time chat app",
    "output": json.dumps({
        "content": "I'll organise this into backend, frontend, and infra.",
        "managers": [
            {"name": "Backend Manager", "job": "Implement WebSocket server."},
            {"name": "Frontend Manager", "job": "Build chat UI with React."},
        ],
    }),
})
demo_raw = f"{demo_valid_line}\nnot valid json\n"

valid, invalid = parse_generated_lines(demo_raw, "director")
print(f"Parsed {len(valid)} valid sample(s), {invalid} invalid.")
if valid:
    # Instruction should have been normalised to the canonical value
    print("Instruction normalised:", valid[0]["instruction"] == NODE_CONFIG["director"]["instruction"])

### 1.8 Orchestration — Generate All Samples

`generate_all` is the top-level async driver that coordinates everything:
1. Iterates over each requested node type.
2. Reads existing JSONL samples to build a deduplication set.
3. Launches concurrent batches using `asyncio.Semaphore` to cap in-flight requests.
4. Tracks stall iterations (no new samples) and a wall-clock time limit to prevent infinite loops.
5. Appends new samples to the JSONL file after generation is complete.

In [ ]:
async def generate_batch(
    session: aiohttp.ClientSession,
    endpoint: str,
    model: str,
    node_type: str,
    batch_size: int,
    existing_inputs: list,
    temperature: float = 0.8,
):
    """Generate one batch of samples and return (valid, invalid_count, error)."""
    prompt = build_generation_prompt(node_type, batch_size, existing_inputs)
    raw, err = await call_tensorrt(session, endpoint, model, prompt, temperature)
    if err:
        return [], 0, err
    valid, invalid = parse_generated_lines(raw, node_type)
    return valid, invalid, None


async def generate_all(args):
    """
    Orchestrate concurrent batch generation for the requested node types.

    Args:
        args: Namespace with fields:
              node_type, count, endpoint, model, concurrency, batch_size, temperature.
    """
    node_types = NODE_TYPES if args.node_type == "all" else [args.node_type]
    os.makedirs(DATA_DIR, exist_ok=True)

    for node_type in node_types:
        print(f"\n{'='*60}")
        print(f"Generating {args.count} samples for: {node_type}")
        print(f"Endpoint: {args.endpoint} | Model: {args.model}")
        print(f"Concurrency: {args.concurrency}, Batch size: {args.batch_size}")
        print(f"{'='*60}")

        output_path = os.path.join(DATA_DIR, f"{node_type}.jsonl")

        # Load existing samples for deduplication
        existing_inputs = set()
        if os.path.isfile(output_path):
            with open(output_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        try:
                            existing_inputs.add(json.loads(line).get("input", ""))
                        except json.JSONDecodeError:
                            pass
            print(f"Existing samples loaded: {len(existing_inputs)}")

        collected     = []
        total_invalid = 0
        start_time    = time.time()
        stall_iters   = 0
        remaining     = args.count
        semaphore     = asyncio.Semaphore(args.concurrency)

        async with aiohttp.ClientSession() as session:
            while remaining > 0:
                # How many concurrent batches fit within the remaining target?
                num_batches = min(
                    args.concurrency,
                    (remaining + args.batch_size - 1) // args.batch_size,
                )
                batch_sizes = [
                    min(args.batch_size, remaining - i * args.batch_size)
                    for i in range(num_batches)
                    if min(args.batch_size, remaining - i * args.batch_size) > 0
                ]

                async def bounded_batch(bs):
                    async with semaphore:
                        return await generate_batch(
                            session, args.endpoint, args.model,
                            node_type, bs, list(existing_inputs), args.temperature,
                        )

                results = await asyncio.gather(*[bounded_batch(s) for s in batch_sizes])

                prev_count = len(collected)
                for valid, invalid, err in results:
                    if err:
                        print(f"  Batch error: {err}")
                        continue
                    total_invalid += invalid
                    for sample in valid:
                        if sample["input"] not in existing_inputs:
                            collected.append(sample)
                            existing_inputs.add(sample["input"])

                remaining = args.count - len(collected)
                elapsed   = time.time() - start_time
                print(f"  Progress: {len(collected)}/{args.count} ({total_invalid} invalid) [{elapsed:.1f}s]")

                # Stall and time-limit guards
                if len(collected) == prev_count:
                    stall_iters += 1
                else:
                    stall_iters = 0

                if stall_iters >= MAX_STALL_ITERATIONS:
                    print(f"  Stalled for {MAX_STALL_ITERATIONS} consecutive rounds. Stopping.")
                    break
                if elapsed > MAX_GENERATION_SECONDS:
                    print(f"  Time limit ({MAX_GENERATION_SECONDS}s) reached. Stopping.")
                    break

        # Append valid samples to JSONL file
        if collected:
            with open(output_path, "a", encoding="utf-8") as f:
                for sample in collected:
                    f.write(json.dumps(sample, ensure_ascii=False) + "\n")
            print(f"\nSaved {len(collected)} samples → {output_path}")
        else:
            print(f"\nNo valid samples generated for {node_type}.")

    print(f"\n{'='*60}\nGeneration complete. Data in {DATA_DIR}/\n{'='*60}")


print("generate_all defined. Example usage (requires TensorRT-LLM server):")
print("  import types")
print("  args = types.SimpleNamespace(node_type='director', count=10, endpoint='http://localhost:8000',")
print("         model='meta-llama/Llama-3.1-70B-Instruct', concurrency=8, batch_size=5, temperature=0.8)")
print("  asyncio.run(generate_all(args))")

### 1.9 CLI Entry Point for `generate_samples.py`

When executed as a script, `argparse` parses command-line arguments. The `--overwrite` flag is useful for regenerating a node type from scratch without appending to an existing JSONL file.

```bash
# From AI/src/fine-tuning/
python scripts/generate_samples.py --node-type director --count 50
python scripts/generate_samples.py --node-type all --count 100 --concurrency 16
python scripts/generate_samples.py --node-type agent --count 200 --overwrite \
    --endpoint http://my-server:8000 --model meta-llama/Llama-3.3-70B-Instruct
```

---
## Part 2 — Fine-Tune with Unsloth FP4

**Script:** `AI/src/fine-tuning/scripts/finetune.py`

Fine-tunes a base model on the generated JSONL data using **Unsloth** (optimised LoRA training) with **4-bit (FP4) quantisation** to minimise VRAM usage.

### What happens during training
1. Loads the base model in FP4 via `FastLanguageModel.from_pretrained`.
2. Applies LoRA adapters (rank 16) to all attention and MLP projection layers.
3. Converts JSONL samples to Alpaca-style prompts (`### Instruction / ### Input / ### Response`).
4. Trains with `SFTTrainer`, AdamW-8bit optimiser, bf16 or fp16 (auto-detected).
5. Saves checkpoints per epoch and final weights to `output/<node-type>/final/`.

### 2.1 Imports and Paths

In [ ]:
# NOTE: Unsloth, transformers, trl, and datasets must be installed.
# These imports will fail unless you are running in a GPU environment
# with the fine-tuning requirements installed:
#   pip install -r AI/src/fine-tuning/requirements.txt

import argparse
import json
import os
import sys

import torch
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel

# ── Resolve paths relative to fine-tuning base directory ────────────────────
FINETUNE_BASE = os.path.abspath("AI/src/fine-tuning")
DATA_DIR_FT   = os.path.join(FINETUNE_BASE, "data-samples")
OUTPUT_BASE   = os.path.join(FINETUNE_BASE, "output")

NODE_TYPES_FT  = ["director", "manager", "supervisor", "agent"]

# ── Model and training constants ─────────────────────────────────────────────
DEFAULT_MODEL_FT = "unsloth/Llama-3.1-8B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH   = 2048  # Maximum token length for training samples
LOAD_IN_4BIT     = True  # Enable FP4 quantisation

print("Fine-tuning imports loaded.")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### 2.2 JSONL Loader and Alpaca Prompt Formatter

Training data is stored in JSONL files with `instruction`, `input`, and `output` fields. The Alpaca prompt template is the format the base model (and the Ollama Modelfile) expect — it structures the context so the model learns exactly when to produce structured JSON output.

In [ ]:
def load_jsonl(filepath: str) -> list[dict]:
    """
    Load a JSONL file and return a list of dicts.

    Args:
        filepath: Absolute or relative path to the .jsonl file.

    Returns:
        List of parsed JSON objects (one per non-empty line).
    """
    records = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def format_prompt(sample: dict) -> str:
    """
    Convert a training sample dict to an Alpaca-style prompt string.

    The Alpaca template uses three delimited sections:
      ### Instruction:  (the system instruction for this node type)
      ### Input:        (the user request or job description)
      ### Response:     (the expected structured JSON output)

    This is the same format used in the Ollama Modelfile, ensuring
    training and inference prompts are aligned.

    Args:
        sample: Dict with 'instruction', 'input', and 'output' keys.

    Returns:
        Formatted prompt string ready for tokenisation.
    """
    instruction = sample.get("instruction", "")
    inp         = sample.get("input", "")
    output      = sample.get("output", "")
    return (
        "### Instruction:\n" + instruction + "\n\n"
        "### Input:\n"       + inp         + "\n\n"
        "### Response:\n"    + output
    )


def prepare_dataset(node_types: list[str]) -> Dataset:
    """
    Load and merge JSONL data for the requested node types into a HuggingFace Dataset.

    Args:
        node_types: List of node types to include (e.g. ['director', 'manager']).

    Returns:
        HuggingFace Dataset with a single 'text' column of Alpaca-formatted strings.

    Raises:
        SystemExit: if no data files are found for any of the requested node types.
    """
    all_records = []
    for nt in node_types:
        path = os.path.join(DATA_DIR_FT, f"{nt}.jsonl")
        if not os.path.isfile(path):
            print(f"Warning: data file not found for '{nt}': {path}")
            continue
        records = load_jsonl(path)
        print(f"Loaded {len(records)} samples from {nt}.jsonl")
        all_records.extend(records)

    if not all_records:
        print("Error: no training data loaded.")
        sys.exit(1)

    texts = [format_prompt(r) for r in all_records]
    return Dataset.from_dict({"text": texts})


# ── Preview — format one sample from director.jsonl ──────────────────────────
director_path = os.path.join(DATA_DIR_FT, "director.jsonl")
if os.path.isfile(director_path):
    sample = load_jsonl(director_path)[0]
    print("=== Alpaca-formatted sample preview ===")
    print(format_prompt(sample)[:500])
else:
    print(f"director.jsonl not found at {director_path}. Run generate_samples.py first.")

### 2.3 Fine-Tuning Loop

The `run_finetune` function executes the six-step training pipeline:

| Step | Action |
|------|--------|
| 1    | Load base model in FP4 via `FastLanguageModel.from_pretrained` |
| 2    | Attach LoRA adapters (rank 16) to all Q/K/V/O + gate/up/down projections |
| 3    | Build the HuggingFace `Dataset` from JSONL files |
| 4    | Configure `TrainingArguments` (bf16/fp16 auto-detected, AdamW-8bit) |
| 5    | Create `SFTTrainer` and start training |
| 6    | Save final merged weights to `output/<node-type>/final/` |

In [ ]:
def run_finetune(args):
    """
    Execute the fine-tuning loop for the specified node type(s).

    Args:
        args: Namespace with fields:
              node_type  (str)  — 'director', 'manager', 'supervisor', 'agent', or 'all'.
              base_model (str)  — Unsloth-compatible HuggingFace model ID.
              epochs     (int)  — Number of training epochs.
    """
    # ── Resolve node types ────────────────────────────────────────────────────
    node_types = NODE_TYPES_FT if args.node_type == "all" else [args.node_type]

    print(f"Node types:  {node_types}")
    print(f"Base model:  {args.base_model}")
    print(f"Epochs:      {args.epochs}")
    print(f"FP4 (4-bit): {LOAD_IN_4BIT}")

    # ── Step 1: Load base model with FP4 quantisation ────────────────────────
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=args.base_model,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
    )

    # ── Step 2: Apply LoRA adapters ───────────────────────────────────────────
    # rank=16 balances adaptation capacity vs. parameter count.
    # We target all attention (Q/K/V/O) and feed-forward (gate/up/down) projections.
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,       # 0 is optimal for Unsloth
        bias="none",
        use_gradient_checkpointing="unsloth",  # Unsloth's memory-efficient checkpointing
    )

    # ── Step 3: Prepare dataset ───────────────────────────────────────────────
    dataset = prepare_dataset(node_types)
    print(f"Total training samples: {len(dataset)}")

    # ── Step 4: Training arguments ────────────────────────────────────────────
    output_dir = os.path.join(OUTPUT_BASE, args.node_type)
    os.makedirs(output_dir, exist_ok=True)

    # Auto-select bf16 (preferred on Ampere+) or fp16 (older GPUs)
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=2,      # Batch size per GPU
        gradient_accumulation_steps=4,      # Effective batch size = 2 × 4 = 8
        warmup_steps=5,
        num_train_epochs=args.epochs,
        learning_rate=2e-4,
        fp16=not use_bf16,
        bf16=use_bf16,
        logging_steps=1,
        save_strategy="epoch",              # Save checkpoint after each epoch
        optim="adamw_8bit",                 # 8-bit AdamW reduces optimiser memory
        seed=42,
    )

    # ── Step 5: Trainer ───────────────────────────────────────────────────────
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        args=training_args,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,  # No sequence packing — each sample is independent
    )

    # ── Step 6: Train and save ────────────────────────────────────────────────
    print("Starting fine-tuning …")
    trainer.train()

    final_dir = os.path.join(output_dir, "final")
    model.save_pretrained(final_dir)
    tokenizer.save_pretrained(final_dir)
    print(f"Fine-tuning complete. Weights saved to {final_dir}")


print("run_finetune defined. Requires a GPU with ≥16 GB VRAM to execute.")
print("Example usage:")
print("  import types")
print("  args = types.SimpleNamespace(node_type='all',")
print("         base_model='unsloth/Llama-3.1-8B-Instruct-bnb-4bit', epochs=3)")
print("  run_finetune(args)")

---
## Part 3 — Export to GGUF for Ollama

**Script:** `AI/src/fine-tuning/scripts/export_gguf.py`

After fine-tuning, the LoRA adapter weights need to be merged back into the base model and then exported to **GGUF** (a compact binary format used by llama.cpp and Ollama). GGUF supports several quantisation schemes that let you trade off model size against response quality:

| Method   | Bits | Approx. File Size | Quality |
|----------|------|-------------------|--------|
| `q4_k_m` | 4    | ~4 GB (8B model)  | Good (recommended default) |
| `q5_k_m` | 5    | ~5 GB             | Better |
| `q8_0`   | 8    | ~9 GB             | High |
| `f16`    | 16   | ~16 GB            | Full precision |

Exported files are written to `AI/src/fine-tuning/export/`.

### 3.1 Imports and Paths

In [ ]:
# Unsloth must be installed (pip install -r AI/src/fine-tuning/requirements.txt)
import argparse
import os
import sys

from unsloth import FastLanguageModel

# ── Paths ─────────────────────────────────────────────────────────────────────
EXPORT_BASE_DIR = os.path.abspath("AI/src/fine-tuning")
EXPORT_DIR      = os.path.join(EXPORT_BASE_DIR, "export")
MAX_SEQ_LEN_EXP = 2048

print(f"Export directory: {EXPORT_DIR}")

### 3.2 GGUF Export Function

Unsloth's `model.save_pretrained_gguf` handles the LoRA merge and llama.cpp quantisation conversion in one call. The `quantization_method` parameter maps directly to llama.cpp quantisation names.

In [ ]:
def export_gguf(args):
    """
    Merge LoRA adapters into the base model and export to GGUF format.

    Args:
        args: Namespace with fields:
              model_dir    (str) — Path to the fine-tuned model directory
                                   (e.g. 'AI/src/fine-tuning/output/all/final').
              quantization (str) — GGUF quantisation method
                                   ('q4_k_m', 'q5_k_m', 'q8_0', or 'f16').

    Raises:
        SystemExit: if the model directory does not exist.
    """
    model_dir = args.model_dir
    if not os.path.isdir(model_dir):
        print(f"Error: model directory not found: {model_dir}")
        sys.exit(1)

    print(f"Loading fine-tuned model from: {model_dir}")

    # Load with the same settings used during fine-tuning
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_dir,
        max_seq_length=MAX_SEQ_LEN_EXP,
        load_in_4bit=True,
    )

    os.makedirs(EXPORT_DIR, exist_ok=True)

    quantization = args.quantization
    print(f"Exporting to GGUF with quantisation method: {quantization}")

    # Merges LoRA → base model, then quantises and saves as .gguf
    model.save_pretrained_gguf(
        EXPORT_DIR,
        tokenizer,
        quantization_method=quantization,
    )

    print(f"GGUF export complete. Files saved to: {EXPORT_DIR}")
    print("\nNext steps:")
    print(f"  1. Copy the Modelfile from {EXPORT_BASE_DIR}/Modelfile into {EXPORT_DIR}/")
    print(f"  2. Update the FROM line in Modelfile to point to the .gguf file")
    print("  3. Run:  ollama create nodpt -f Modelfile")
    print("  4. Run:  ollama run nodpt")


print("export_gguf defined. Requires a GPU and the fine-tuned model directory.")
print("Example usage:")
print("  import types")
print("  args = types.SimpleNamespace(")
print("      model_dir='AI/src/fine-tuning/output/all/final',")
print("      quantization='q4_k_m')")
print("  export_gguf(args)")

---
## Part 4 — Deploy with Ollama

After exporting to GGUF, deploy the model locally with Ollama.

### Step 1 — Verify the GGUF file

```bash
ls -lh AI/src/fine-tuning/export/*.gguf
```

### Step 2 — Inspect the Modelfile

The `Modelfile` template uses the Alpaca prompt format, matching how the model was trained:

In [ ]:
modelfile_path = os.path.join(os.path.abspath("AI/src/fine-tuning"), "Modelfile")
if os.path.isfile(modelfile_path):
    with open(modelfile_path, "r") as f:
        print(f.read())
else:
    print(f"Modelfile not found at {modelfile_path}")

### Step 3 — Create and run the Ollama model

```bash
# From the repository root:
ollama create nodpt -f AI/src/fine-tuning/Modelfile
ollama run nodpt
```

### Step 4 — Test with run.py

```bash
cd AI/src
python run.py director --prompt sample.txt --model nodpt
python run.py agent --prompt-text "Write a REST endpoint for user signup" --model nodpt
```

---

## Summary

| Step | Script | Input | Output |
|------|--------|-------|--------|
| 1. Generate data | `generate_samples.py` | TensorRT-LLM (70B+) | `data-samples/*.jsonl` |
| 2. Fine-tune     | `finetune.py`          | JSONL + base model  | `output/<type>/final/` |
| 3. Export GGUF   | `export_gguf.py`       | Fine-tuned weights  | `export/*.gguf` |
| 4. Deploy        | Ollama CLI             | GGUF + Modelfile    | `ollama run nodpt` |

### Troubleshooting

| Problem | Fix |
|---------|-----|
| `CUDA out of memory` | Reduce `per_device_train_batch_size` or `max_seq_length` in `finetune.py` |
| `No module named 'unsloth'` | `pip install -r AI/src/fine-tuning/requirements.txt` |
| GGUF file too large | Use `q4_k_m` instead of `f16` |
| Model produces bad JSON | Add more training data and re-run with more epochs |
| `ollama create` fails | Ensure the `FROM` path in Modelfile points to an existing `.gguf` file |